In [56]:
import json
import os
import re

In [57]:
with open('llama3_8b_judge_new_sample.jsonl', 'r') as f:
    llama3_8b = [json.loads(line) for line in f]

with open('qwen_8b_judge_new_sample.jsonl', 'r') as f:
    qwen = [json.loads(line) for line in f]

with open('r1_llama_8b_judge_processed.jsonl', 'r') as f:
    r1_llama3_8b = [json.loads(line) for line in f]

In [58]:
# Remove all features except Input.full_text and llm_response
llama3_8b = [{k: v for k, v in item.items() if k in ['Input.full_text', 'llm_response']} for item in llama3_8b]
qwen = [{k: v for k, v in item.items() if k in ['Input.full_text', 'llm_response']} for item in qwen]
r1_llama3_8b = [{k: v for k, v in item.items() if k in ['Input.full_text', 'llm_response']} for item in r1_llama3_8b]


In [59]:
llama3_8b

[{'Input.full_text': 'Happy to aid in the fight against germany and france as well',
  'llm_response': '1. NO\n2. NO\n3. YES (REASSURANCE)\n4. NO\n5. YES\n6. NO\n7. NO\n8. NO'},
 {'Input.full_text': "I'm going to be hit or miss with connectivity this evening, so I'm putting moves in now.",
  'llm_response': '1. YES\n2. NO\n3. NO\n4. NO\n5. NO\n6. NO\n7. YES\n8. NO'},
 {'Input.full_text': 'And if so, how do you feel about it?',
  'llm_response': '1. NO\n2. NO\n3. YES (APOLOGIES)\n4. NO\n5. YES\n6. YES\n7. YES (PERSONAL THOUGHTS)\n8. NO'},
 {'Input.full_text': "I'd like to hold it for at least one more season, but then will probably move out",
  'llm_response': '1. NO\n2. NO\n3. NO\n4. NO\n5. NO\n6. NO\n7. YES\n8. NO'},
 {'Input.full_text': 'I have so much to offer besides just bedside adventures',
  'llm_response': '1. NO\n2. NO\n3. YES (PERSONAL THOUGHTS)\n4. NO\n5. NO\n6. NO\n7. YES\n8. NO'},
 {'Input.full_text': "I really don't think an attack from Gascony would be an issue, but I un

In [60]:
def parse_llm_response(response):
    """
    解析llm的response为一个map，key为序号（int），value为yes/no的1/0
    例如: "1. yes\n2. no\n3. yes" -> {1: 1, 2: 0, 3: 1}
    如果答案数不足8个，则返回空dict

    新增：如果response中包含</think>，则只解析</think>之后的内容
    """
    # 如果有</think>，只取其后的内容
    if "</think>" in response:
        response = response.split("</think>", 1)[1]
    result = {}
    lines = response.strip().split('\n')
    for line in lines:
        m = re.match(r'(\d+)\.\s*(yes|no)', line.strip(), re.IGNORECASE)
        if m:
            idx = int(m.group(1))
            val = 1 if m.group(2).lower() == 'yes' else 0
            result[idx] = val
    if len(result) != 8:
        return {}
    return result

def process_single_result(result):
    """
    对单条result进行解析和处理，返回处理后的dict（如果无效则返回None）
    """
    llm_response_mapping = {
        1: "1gamemove.yes",
        2: "2reasoning.yes", 
        3: "3rapport.yes",
        4: "3a_apologies.yes",
        5: "3a_compliment.yes",
        6: "3a_personalthoughts.yes",
        7: "3a_reassurance.yes",
        8: "4shareinformation.yes"
    }
    parsed = parse_llm_response(result)
    if parsed and len(parsed) == 8:
        result_dict = {}
        for i in range(1, 9):
            result_dict[llm_response_mapping[i]] = parsed[i]
        return result_dict
    else:
        # print(f"Invalid response format or not 8 items: {result}")
        # print(f"Parsed: {parsed}")
        # print(f"Skipping this item")
        # print(f"--------------------------------")
        return None

def add_parsed_llm_response(data_list):
    processed_list = []
    for item in data_list:
        parsed = process_single_result(item['llm_response'])
        if parsed:
            item.update(parsed)
            del item['llm_response']
            processed_list.append(item)
    return processed_list

In [61]:
llama3_8b = add_parsed_llm_response(llama3_8b)
qwen = add_parsed_llm_response(qwen)
r1_llama3_8b = add_parsed_llm_response(r1_llama3_8b)

In [62]:
len(llama3_8b), len(qwen), len(r1_llama3_8b)

(113, 128, 126)

In [65]:
from statsmodels.stats.inter_rater import fleiss_kappa
import numpy as np

# --- 1. Keep the intersection and make its order explicit -------------
common_texts = sorted(
      set(d['Input.full_text'] for d in llama3_8b)
    & set(d['Input.full_text'] for d in qwen)
    & set(d['Input.full_text'] for d in r1_llama3_8b)
)

# --- 2. Helper to convert any truthy/falsey form to 0 / 1 -------------
def to_int(label):
    if isinstance(label, bool):
        return int(label)
    return 1 if str(label).lower() in {"yes", "true", "1"} else 0

# --- 3. Dict for O(1) lookup instead of repeated list scans ----------
def key_dict(lst):
    return {d["Input.full_text"]: d for d in lst}

d_llama   = key_dict(llama3_8b)
d_qwen    = key_dict(qwen)
d_r1      = key_dict(r1_llama3_8b)

# --- 4. Build the rating matrix --------------------------------------
categories = [
    "1gamemove.yes", "2reasoning.yes", "3rapport.yes", "3a_apologies.yes",
    "3a_compliment.yes", "3a_personalthoughts.yes", "3a_reassurance.yes",
    "4shareinformation.yes"
]

rows = []
for text in common_texts:
    rec1, rec2, rec3 = d_llama[text], d_qwen[text], d_r1[text]
    for cat in categories:
        cnt = [0, 0]                       # [# “no”, # “yes”]
        for rec in (rec1, rec2, rec3):
            cnt[to_int(rec[cat])] += 1
        rows.append(cnt)

ratings = np.asarray(rows, dtype=int)

# --- 5. Overall Fleiss’ κ -------------------------------------------
overall_kappa = fleiss_kappa(ratings)
print(f"Overall Fleiss' kappa: {overall_kappa:.3f}")

# --- 6. Per-feature κ -------------------------------------------------
k_per_cat = {}
blocks = len(categories)
for i, cat in enumerate(categories):
    block = ratings[i::blocks]             # every ‘blocks’-th row
    k_per_cat[cat] = fleiss_kappa(block)
    print(f"{cat:25s}: κ = {k_per_cat[cat]:.3f}")


Overall Fleiss' kappa: 0.363
1gamemove.yes            : κ = 0.364
2reasoning.yes           : κ = 0.248
3rapport.yes             : κ = -0.022
3a_apologies.yes         : κ = 0.308
3a_compliment.yes        : κ = 0.350
3a_personalthoughts.yes  : κ = 0.519
3a_reassurance.yes       : κ = 0.059
4shareinformation.yes    : κ = 0.371
